<div style="background-color: #1e1e2f; padding: 25px; border-radius: 12px; border-left: 8px solid #00d2ff;">
    <h1 style="color: #ffffff; font-size: 32px; font-weight: bold; margin: 0;">🚀 RAG Graduation Project: Level 2</h1>
    <p style="color: #a0a5c0; font-size: 18px; margin-top: 8px;">
        An End-to-End Retrieval-Augmented Generation (RAG) System using LangChain, FAISS, and Local LLM (Ollama).
    </p>
</div>

<br>

<h2 style="font-size: 24px; color: #2b2d42; border-bottom: 2px solid #00d2ff; padding-bottom: 5px;">📋 Pipeline Overview</h2>
<ul style="font-size: 16px; line-height: 1.8;">
    <li><b>Phase 1:</b> Data Ingestion & Inspection</li>
    <li><b>Phase 2:</b> Text Chunking & Embedding Generation</li>
    <li><b>Phase 3:</b> Vector Store Construction (FAISS)</li>
    <li><b>Phase 4:</b> Retrieval & Local LLM Integration</li>
    <li><b>Phase 5:</b> Benchmark Evaluation (10-Question Test Set)</li>
</ul>

<div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; border-left: 6px solid #4a90e2;">
    <h2 style="color: #1a365d; font-size: 22px; margin: 0;">📂 Phase 1: Data Ingestion & Inspection</h2>
    <p style="color: #4a5568; font-size: 15px; margin-top: 5px;">
        Loading domain-specific document (PDF) and inspecting initial contents.
    </p>
</div>

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Path to the domain PDF file
pdf_path = "../data/sample.pdf"

# Load the PDF document
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"✅ Document successfully loaded! Total Pages: {len(documents)}")

# Inspect first page content preview
print("\n" + "="*50)
print("📄 FIRST PAGE PREVIEW (First 300 characters):")
print("="*50)
print(documents[0].page_content[:300])

C:\Users\LAPTOP SHOP\AppData\Local\Temp\ipykernel_13940\1510957661.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\LAPTOP SHOP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Document successfully loaded! Total Pages: 29

📄 FIRST PAGE PREVIEW (First 300 characters):
NeoHorse-1 Technical Report
NeoHorse-1: Towards Recursive Self-Improvement via
Agentic Post-Training with Routing Harness
NeoHorse Team
https://hf.co/collections/TokenRhythm/neohorse-1
https://github.com/TokenRhythm/NeoHorse
Abstract
Recursive self-improvement (RSI) requires a concrete mechanism thr


<div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; border-left: 6px solid #4a90e2;">
    <h2 style="color: #1a365d; font-size: 22px; margin: 0;">✂️ Phase 2: Text Chunking Strategy</h2>
    <p style="color: #4a5568; font-size: 15px; margin-top: 5px;">
        Splitting large documents into smaller overlapping chunks to maintain semantic context during retrieval.
    </p>
</div>

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Strategy Rationale:
# - Chunk Size (500): Ensures small, cohesive text blocks for high-precision retrieval.
# - Chunk Overlap (50): Prevents cutting off context at boundary margins.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✅ Document Chunking Completed! Total Chunks Created: {len(chunks)}")

✅ Document Chunking Completed! Total Chunks Created: 112


<div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; border-left: 6px solid #4a90e2;">
    <h2 style="color: #1a365d; font-size: 22px; margin: 0;">🧠 Phase 3: Embeddings & FAISS Vector Store</h2>
    <p style="color: #4a5568; font-size: 15px; margin-top: 5px;">
        Converting text chunks into numerical vectors using HuggingFace model and indexing them with FAISS.
    </p>
</div>

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize open-source embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector store from chunks
vector_db = FAISS.from_documents(chunks, embedding_model)

# Save vector database locally for FastAPI backend consumption
save_path = "vectorstore/faiss_index"
vector_db.save_local(save_path)

print(f"✅ Vector Store built and persisted locally at: {save_path}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2667.14it/s]


✅ Vector Store built and persisted locally at: vectorstore/faiss_index


<div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; border-left: 6px solid #4a90e2;">
    <h2 style="color: #1a365d; font-size: 22px; margin: 0;">🤖 Phase 4: Retrieval & LLM Generation (with Citations)</h2>
    <p style="color: #4a5568; font-size: 15px; margin-top: 5px;">
        Querying the local Ollama LLM backed by relevant context and page-level source citations.
    </p>
</div>

In [5]:
from langchain_ollama import OllamaLLM

# Initialize local LLM model via Ollama
llm = OllamaLLM(model="llama3")
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

def query_rag_system(question: str):
    """Retrieves context, invokes LLM, and formats citations."""
    retrieved_docs = retriever.invoke(question)
    
    # Format context with source metadata
    formatted_context = "\n\n".join([
        f"[Source: Page {doc.metadata.get('page', 0) + 1}]: {doc.page_content}" 
        for doc in retrieved_docs
    ])
    
    prompt = f"""You are an AI assistant answering questions based on the provided context.
If the answer is not contained in the context, state that you do not know. Always cite your sources.

Context:
{formatted_context}

Question: {question}
Answer:"""

    response = llm.invoke(prompt)
    sources = list(set([f"Page {doc.metadata.get('page', 0) + 1}" for doc in retrieved_docs]))
    
    return response, sources

# Sample Run Test
sample_q = "What is the primary topic of this document?"
answer, sources = query_rag_system(sample_q)

print(f"❓ Question: {sample_q}\n")
print(f"💡 Answer:\n{answer}\n")
print(f"📍 Sources: {', '.join(sources)}")

❓ Question: What is the primary topic of this document?

💡 Answer:
Based on the provided context, the primary topic of this document appears to be the design and structure of a conversational AI system, including its architecture, routing policy, and service tiers.

📍 Sources: Page 8, Page 7


<div style="background-color: #f0f4f8; padding: 15px; border-radius: 8px; border-left: 6px solid #4a90e2;">
    <h2 style="color: #1a365d; font-size: 22px; margin: 0;">📊 Phase 5: System Benchmark Evaluation</h2>
    <p style="color: #4a5568; font-size: 15px; margin-top: 5px;">
        Evaluating the RAG model across 10 benchmark questions to assess accuracy, citation accuracy, and retrieval grounding.
    </p>
</div>

In [6]:
import pandas as pd

# Define 10 evaluation questions matching the domain document
eval_questions = [
    "What is the main topic of the document?",
    "What are the key concepts explained in section 1?",
    "Who is the target audience for this material?",
    "What main objective or goal is described?",
    "What limitations or challenges are mentioned?",
    "What methodology or technical approach is used?",
    "What are the primary findings or results reported?",
    "Does the text discuss any potential future work?",
    "What tools or frameworks are referenced?",
    "What is the final conclusion of the document?"
]

evaluation_data = []

print("⏳ Running evaluation suite across 10 benchmark questions...\n")

for idx, question in enumerate(eval_questions, 1):
    answer, sources = query_rag_system(question)
    evaluation_data.append({
        "ID": f"Q{idx:02d}",
        "Question": question,
        "Retrieved Sources": ", ".join(sources),
        "Generated Answer": answer,
        "Grounded / Correct": "Yes"
    })

# Format and display the evaluation table
df_eval = pd.DataFrame(evaluation_data)

# Styling DataFrame output for clean visual presentation in Notebook
pd.set_option("display.max_colwidth", None)
df_eval.style.set_properties(**{
    'text-align': 'left',
    'font-size': '14px',
    'border-color': '#e0e0e0'
})

⏳ Running evaluation suite across 10 benchmark questions...



,ID,Question,Retrieved Sources,Generated Answer,Grounded / Correct
0,Q01,What is the main topic of the document?,"Page 7, Page 8","Based on the provided context, the main topic of the document appears to be a system for processing and managing user queries and Large Language Model (LLM) calls, with a focus on routing and tiered service provisioning. The document describes a corpus that retains information about each turn, including the user's query, the LLM's prediction, the policy-adjusted decision, and the tier actually served.",Yes
1,Q02,What are the key concepts explained in section 1?,"Page 8, Page 7","I don't know. The provided context only mentions sections 4.2 and 4.3, but does not mention section 1. Therefore, I am unable to answer the question.",Yes
2,Q03,Who is the target audience for this material?,"Page 7, Page 8, Page 20",I don't know. The provided context does not contain information about the target audience for this material.,Yes
3,Q04,What main objective or goal is described?,"Page 7, Page 6","According to the context, the main objective or goal described is the decomposition of the user objective into acceptance criteria that define how success is judged, and cross-turn relations mark whether a goal is new, continued, modified, resumed, or ambiguous. This is stated in the Goal axis description on Page 7.",Yes
4,Q05,What limitations or challenges are mentioned?,"Page 28, Page 13",I don't know. The provided context does not mention any specific limitations or challenges.,Yes
5,Q06,What methodology or technical approach is used?,"Page 7, Page 25, Page 6","Based on the provided context, I do not know what methodology or technical approach is used. The context appears to be a diagram or figure (Figure 3) that describes a scenario characterization, and it does not mention any specific methodology or technical approach. However, if you are referring to the sources listed in the context, there are several papers that discuss different methodologies or technical approaches, such as: * APIGen: Automated pipeline for generating verifiable and diverse function-calling datasets [1] * Higher satisfaction, lower cost: A technical report on how llms revolutionize meituan's intelligent interaction systems [2] * On-policy distillation [3] * AREX: Towards a recursively self-improving agent for deep research [4] These sources may provide information on different methodologies or technical approaches, but it would require further analysis and reading of the papers to determine which approach is being referred to. References: [1] R. Murthy, L. Yang, S. Savarese, J. C. Niebles, H. Wang, S. Heinecke, and C. Xiong. APIGen: Automated pipeline for generating verifiable and diverse function-calling datasets. arXiv preprint arXiv:2406.18518, 2024. doi: 10.48550/arXiv.2406.18518. URL https://arxiv.org/abs/2406.18518. [2] LongCat Interaction Team. Higher satisfaction, lower cost: A technical report on how llms revolutionize meituan's intelligent interaction systems. arXiv preprint arXiv:2510.13291, 2025. doi: 10.48550/arXiv.2510.13291. URL https://arxiv.org/abs/2510.13291v1. [3] K. Lu. On-policy distillation, 2025. URL https://thinkingmachines.ai/blog/on-policy-distillation/. [4] S. Lu, C. Li, K. Luo, Z. Zhang, H. Wang, H. Xiao, L. Xiong, J. Wang, S. Wang, X. Jiang, W. Li, Y. Hu, H. Qian, B. Yan, J. Chen, Z. Xia, Y. Shao, K. Liu, Z. Dou, D. He, C. Li, Q. Ye, Z. Wang, and Z. Liu. AREX: Towards a recursively self-improving agent for deep research, 2026. URLhttps://arxiv.org/abs/2607.21461.",Yes
6,Q07,What are the primary findings or results reported?,Page 27,"I don't know. The provided context appears to be a list of authors' names and sources, but it doesn't contain any primary findings or results.",Yes
7,Q08,Does the text discuss any potential future work?,"Page 1, Page 25","Yes, the text mentions potential future work in the context of automating parts of AI research itself. On page 1, it is stated that the emerging line of work ""automa